# English Multi-Speaker FastSpeech 2 — Kaggle Training Notebook

This notebook executes full FastSpeech 2 multi-speaker training on the **Svarah English** dataset using GPU T4 x2.
Designed for **Save Version -> Save & Run All** workflow with memory management, strict checkpoint retention (Top 3 best + 1 latest resumable), disk-space safeguards, auto-resume, and direct maintenance of the `English_TTS_Best_Model` folder (without creating an inference ZIP).


In [ ]:
# Cell 1: Environment Initialization
import os
import sys
import platform
import subprocess
import shutil
import torch
try:
    import psutil
except ImportError:
    psutil = None

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

print("=" * 70)
print("CELL 1: ENVIRONMENT & RESOURCE INITIALIZATION")
print("=" * 70)
print(f"OS Platform:       {platform.platform()}")
print(f"Python Version:    {sys.version.split()[0]}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version:      {torch.version.cuda}")
    print(f"cuDNN Version:     {torch.backends.cudnn.version()}")
print(f"Working Directory: {os.getcwd()}")
disk = shutil.disk_usage("/kaggle/working" if os.path.exists("/kaggle/working") else ".")
print(f"Disk Space Free:   {disk.free / (1024**3):.2f} GB / {disk.total / (1024**3):.2f} GB ({disk.free/disk.total*100:.1f}% free)")
if psutil:
    vm = psutil.virtual_memory()
    print(f"System RAM:        {vm.used / (1024**3):.2f} GB used / {vm.total / (1024**3):.2f} GB total ({vm.percent}%)")
print("=" * 70)


In [ ]:
# Cell 2: GPU Configuration & Multi-GPU Setup
import os
import torch

print("=" * 70)
print("CELL 2: GPU CONFIGURATION")
print("=" * 70)

if not torch.cuda.is_available():
    print("[WARNING] No GPU detected! Running on CPU mode.")
    device = torch.device("cpu")
else:
    gpu_count = torch.cuda.device_count()
    print(f"Detected {gpu_count} GPU device(s):")
    for i in range(gpu_count):
        prop = torch.cuda.get_device_properties(i)
        mem_gb = prop.total_memory / (1024**3)
        print(f"  GPU [{i}]: {prop.name} | Total VRAM: {mem_gb:.2f} GB | Compute: {prop.major}.{prop.minor}")

    if gpu_count >= 2:
        print("\n[Configuration] Dual-GPU environment detected (Kaggle T4 x2).")
        print("FastSpeech 2 DataParallel will utilize both GPUs automatically.")
        os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    else:
        print("\n[Configuration] Single GPU detected. Operating on GPU 0.")
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    device = torch.device("cuda")

print(f"Active Device: {device}")
print("=" * 70)


In [ ]:
# Cell 3: Fast Dataset Preparation & Checkpoint Auto-Discovery
import os
import sys
import glob
import shutil
import zipfile
import subprocess
import time

print("=" * 70)
print("CELL 3: INSTANT DATASET PREPARATION & CHECKPOINT AUTO-DISCOVERY")
print("=" * 70)

target_pkg_dir = "/kaggle/working/English_FastSpeech2_Kaggle"

# Check if target directory is already present and complete
prep_present = False
if os.path.exists(target_pkg_dir):
    try:
        sys.path.insert(0, target_pkg_dir)
        from prepare_dataset import verify_dataset
        valid, msg = verify_dataset(target_pkg_dir)
        if valid:
            prep_present = True
            print("[Instant Ready] Verified existing dataset in /kaggle/working.")
    except Exception:
        prep_present = False

if not prep_present:
    # 1. Check for uncompressed directory in /kaggle/input (instant symlink path)
    dir_candidates = glob.glob("/kaggle/input/**/preprocessed_data", recursive=True)
    if dir_candidates:
        src_dir = os.path.dirname(dir_candidates[0])
        print(f"[Fast Symlink] Found input directory at: {src_dir}")
        os.makedirs(target_pkg_dir, exist_ok=True)
        t0 = time.time()
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            t_item = os.path.join(target_pkg_dir, item)
            if item == "preprocessed_data":
                if os.path.islink(t_item) or os.path.exists(t_item):
                    if os.path.islink(t_item): os.unlink(t_item)
                    else: shutil.rmtree(t_item)
                try:
                    os.symlink(s_item, t_item)
                    print(f"  [Instant Symlink] preprocessed_data -> {s_item} (0 MB copied)")
                except Exception as e:
                    print(f"  [Symlink Fallback] Symlink failed ({e}), copying...")
                    shutil.copytree(s_item, t_item, dirs_exist_ok=True)
            elif os.path.isdir(s_item):
                shutil.copytree(s_item, t_item, dirs_exist_ok=True)
            else:
                shutil.copy2(s_item, t_item)
        print(f"[Fast Ready] Setup completed in {time.time() - t0:.2f} seconds!")
    else:
        # 2. Check for zip archive
        zip_candidates = glob.glob("/kaggle/input/**/*English*.zip", recursive=True)
        if not zip_candidates:
            zip_candidates = glob.glob("/kaggle/input/*/*.zip")
        if zip_candidates:
            src_zip = zip_candidates[0]
            print(f"[Extracting] Found archive: {src_zip}")
            unzip_cmd = shutil.which("unzip")
            if unzip_cmd:
                subprocess.run([unzip_cmd, "-q", "-o", src_zip, "-d", "/kaggle/working/"], check=True)
            else:
                with zipfile.ZipFile(src_zip, "r") as zf:
                    zf.extractall("/kaggle/working/")

sys.path.insert(0, target_pkg_dir)
from prepare_dataset import prepare_dataset
prepare_dataset(target_pkg_dir)

# Initialize /kaggle/working/English/checkpoints from discovered checkpoints
out_ckpt_dir = "/kaggle/working/English/checkpoints"
os.makedirs(out_ckpt_dir, exist_ok=True)
candidate_ckpts = []
pkg_ckpt_dir = os.path.join(target_pkg_dir, "checkpoints")
if os.path.exists(pkg_ckpt_dir):
    candidate_ckpts.append(pkg_ckpt_dir)
for d in glob.glob("/kaggle/input/**/checkpoints", recursive=True):
    if d not in candidate_ckpts:
        candidate_ckpts.append(d)

for ckpt_src in candidate_ckpts:
    latest_src = os.path.join(ckpt_src, "latest_checkpoint.pth")
    if os.path.isfile(latest_src) and not os.path.isfile(os.path.join(out_ckpt_dir, "latest_checkpoint.pth")):
        print(f"[Checkpoint Setup] Copying pre-existing checkpoints from {ckpt_src}...")
        for fn in os.listdir(ckpt_src):
            s_f = os.path.join(ckpt_src, fn)
            d_f = os.path.join(out_ckpt_dir, fn)
            if not os.path.exists(d_f):
                if os.path.isdir(s_f):
                    shutil.copytree(s_f, d_f)
                else:
                    shutil.copy2(s_f, d_f)
                print(f"  -> Ready: {fn} ({os.path.getsize(d_f)/(1024**2):.1f} MB)")
        break

if os.path.isfile(os.path.join(out_ckpt_dir, "latest_checkpoint.pth")):
    print(f"\n[Resume Status] Checkpoint ready in {out_ckpt_dir}! Training will AUTO-RESUME seamlessly.")
else:
    print(f"\n[Resume Status] Fresh start (no existing checkpoint in {out_ckpt_dir}). Starting from Step 1.")

os.chdir(target_pkg_dir)
print(f"Current Working Directory: {os.getcwd()}")
print("=" * 70)


In [ ]:
# Cell 4: Dependency and Configuration Verification
import os
import sys
import yaml
import subprocess

print("=" * 70)
print("CELL 4: DEPENDENCY & CONFIGURATION VERIFICATION")
print("=" * 70)

req_file = "requirements.txt"
if os.path.exists(req_file):
    print("[Packages] Installing / verifying dependencies from requirements.txt...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", req_file, "--quiet"], check=True)
    print("[Packages] Dependencies verified.")

config_file = "config.yaml"
assert os.path.isfile(config_file), f"Missing {config_file}!"
with open(config_file, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
print(f"[Config] Loaded {config_file} successfully.")
print(f"  Dataset:       {cfg.get('dataset')}")
print(f"  Sample Rate:   {cfg.get('preprocessing', {}).get('audio', {}).get('sampling_rate')} Hz")
print(f"  Total Steps:   {cfg.get('train', {}).get('step', {}).get('total_step'):,}")
print(f"  Batch Size:    {cfg.get('train', {}).get('optimizer', {}).get('batch_size')}")

vocoder_weight = os.path.join("hifigan", "generator_universal.pth.tar")
assert os.path.isfile(vocoder_weight), f"Missing vocoder weights at {vocoder_weight}!"
voc_size = os.path.getsize(vocoder_weight)
print(f"[Vocoder] HiFi-GAN universal vocoder found ({voc_size / (1024**2):.2f} MB).")
print("=" * 70)


In [ ]:
# Cell 5: Model Initialization and Training Execution
import os
import sys
import subprocess
import shutil

print("=" * 70)
print("CELL 5: MODEL INITIALIZATION & TRAINING LAUNCH")
print("=" * 70)

disk = shutil.disk_usage("/kaggle/working")
print(f"Pre-training Disk Free: {disk.free / (1024**3):.2f} GB")

cmd = [
    "python", "-u", "train.py",
    "--config", "config.yaml",
    "--data_dir", "/kaggle/working/English_FastSpeech2_Kaggle/preprocessed_data",
    "--output_dir", "/kaggle/working/English",
    "--export_dir", "/kaggle/working/English_TTS_Best_Model",
    "--save_step", "1000",
    "--val_step", "1000",
    "--log_step", "100",
    "--batch_size", "16",
    "--total_step", "100000",
    "--resume"
]

print("Executing command:")
print(" ".join(cmd))
print("=" * 70 + "\n")

proc = subprocess.run(cmd)
if proc.returncode != 0:
    raise RuntimeError(f"Training exited with error code {proc.returncode}!")

print("\n" + "=" * 70)
print("Training execution finished successfully.")
print("=" * 70)


In [ ]:
# Cell 6: Best Model Directory Verification (No Zip Created)
import os
import sys

print("=" * 70)
print("CELL 6: BEST MODEL DIRECTORY VERIFICATION")
print("=" * 70)

best_model_dir = "/kaggle/working/English_TTS_Best_Model"
print(f"Verifying inference directory: {best_model_dir}\n")

required_items = [
    "best_model.pth",
    "config.yaml",
    "stats.json",
    "speakers.json",
    "inference.py",
    "hifigan"
]

for item in required_items:
    path = os.path.join(best_model_dir, item)
    exists = os.path.exists(path)
    sz_str = f"({os.path.getsize(path)/(1024**2):.2f} MB)" if (exists and os.path.isfile(path)) else ""
    print(f"  [{'OK' if exists else 'MISSING'}] {item:<25} {sz_str}")

print("\n[Retention & Export Policy]")
print("  - Checkpoints: Top 3 best validation models + 1 latest resumable checkpoint.")
print("  - Export: English_TTS_Best_Model folder maintained directly without creating an inference ZIP archive.")
print("=" * 70)


In [ ]:
# Cell 7: Final Execution Summary & Audit
import os
import glob
import json
import torch

print("=" * 70)
print("CELL 7: FINAL EXECUTION SUMMARY & RETENTION AUDIT")
print("=" * 70)

ckpt_dir = "/kaggle/working/English/checkpoints"
export_dir = "/kaggle/working/English_TTS_Best_Model"
index_file = os.path.join(ckpt_dir, "checkpoints_index.json")
latest_ckpt_file = os.path.join(ckpt_dir, "latest_checkpoint.pth")

final_step = "Unknown"
best_loss = "Unknown"
epoch = "Unknown"
is_valid = False

if os.path.isfile(latest_ckpt_file):
    try:
        ckpt = torch.load(latest_ckpt_file, map_location="cpu", weights_only=False)
        final_step = ckpt.get("step", "Unknown")
        epoch = ckpt.get("epoch", "Unknown")
        bl = ckpt.get("best_loss", "Unknown")
        best_loss = f"{bl:.4f}" if isinstance(bl, float) else str(bl)
        is_valid = True
    except Exception as e:
        print(f"[Error] Failed to load latest checkpoint: {e}")

# Load index
best_list = []
if os.path.isfile(index_file):
    try:
        with open(index_file, "r") as f:
            idx_data = json.load(f)
            best_list = idx_data.get("best_checkpoints", [])
    except Exception as e:
        print(f"[Notice] Could not parse index: {e}")

exported_files = sorted(os.listdir(export_dir)) if os.path.exists(export_dir) else []

status_str = "SUCCESS / COMPLETED" if (is_valid and isinstance(final_step, int) and final_step >= 100000) else "RESUMABLE / IN PROGRESS"
print(f"Training Status:       {status_str}")
print(f"Latest Resumable Step: {final_step}")
print(f"Final Epoch:           {epoch}")
print(f"Best Validation Loss:  {best_loss}")
print(f"Retained Best Models:  {len(best_list)} in index (Target: Top 3)")
print("-" * 70)
print("RANKED TOP 3 BEST CHECKPOINTS:")
print(f"  {'Rank':<6} {'Step':<10} {'Val Loss':<12} {'Filename':<28}")
for b in best_list:
    print(f"  {b.get('rank', '-'):<6} {b.get('step', '-'):<10} {b.get('val_loss', 0.0):<12.4f} {b.get('filename', '-'):<28}")
print("-" * 70)
print(f"English_TTS_Best_Model: {len(exported_files)} items in {export_dir}")
for ef in exported_files:
    fpath = os.path.join(export_dir, ef)
    if os.path.isfile(fpath):
        sz = os.path.getsize(fpath)
        print(f"  -> {ef:<32} {sz:>12,} bytes ({sz/(1024**2):.2f} MB)")
    else:
        print(f"  -> {ef:<32} [directory]")
print("=" * 70)
if not is_valid:
    raise RuntimeError("Latest checkpoint verification failed!")
print("\n[READY] Training run status verified. Top 3 best checkpoints + 1 latest checkpoint preserved.\n")
